# Train the bud model
This notebook (re)trains the model that assigns buds to mothers automatically (`bud_model.py`, "model D").
You only need it to train on **your own** corrected images; the trained models that come with buddy were made the same way.

**What it learns from:** image folders you corrected in step 4 of `01_fish_bulk_processing.ipynb`
(`corrected/DIC_corrected_masks.tif` + `corrected/bud_corrected_masks.tif`), plus `DIC_masks_pp.tif` (and `DIC_masks.tif`),
`DIC.tif`, `DAPI.tif` and – for the `with_ASH1CLB2` model – `ASH1CLB2.tif` in the same folder.

**One training example = one Omnipose cell:** the crop around it with a soft dot on its centre. The model learns to outline
the cell, its bud (if any) and to say whether it is unbudded / budded / itself a bud. Small uncurated Omnipose objects that
touch a cell are not graded (they may be buds the curation missed). Late-M cells are kept in training.

**Steps:** 1. settings · 2. prepare the images (once) · 3. train · 4. check the result on images the model never saw.

A GPU is strongly recommended (30 epochs ≈ 30 min on a GPU, many hours on a CPU). The first run downloads ImageNet weights once.


In [ ]:
from glob import glob
from pathlib import Path

from buddy.buds import bud_training as bt
from buddy.buds.bud_model import BudModel


## 1. Settings

In [ ]:
# image folders = the output folders of the bulk-processing notebook (one folder per image)
FOLDERS = sorted(glob(r"F:\BUD analyses\*\*\output\*"))

MODEL = "with_ASH1CLB2"            # "DIC_DAPI_only" or "with_ASH1CLB2"
ASH_CHANNEL = "ASH1CLB2"           # channel name of the ASH1/CLB2 smFISH
EXCLUDE_STRAINS = ["YET563", "YET572"] if MODEL == "with_ASH1CLB2" else []   # she2/she3: ASH1 mRNA not in the bud

# images kept apart to check the model afterwards (never used for training): folder names containing any of these
TEST_IMAGES = ["BR5"]

EPOCHS = 30                        # weights are also saved every 2 epochs: the last epoch is not always the best (see step 4)
DATA_DIR = Path("bud_training_data")                    # prepared training files go here
OUT_FILE = Path("models") / f"my_bud_model_{MODEL}.pt"  # your trained weights (the shipped models are not overwritten)
USE_GPU = True
print(len(FOLDERS), "image folders found")

## 2. Prepare the images (only once; already prepared images are skipped)
Every corrected image becomes one training file: the channels moved into the frame of the corrected masks, plus a table with one row per Omnipose cell (unbudded / budded / bud / fragment / uncurated). Only the first three kinds are trained on.


In [ ]:
images = []
for folder in FOLDERS:
    if not (Path(folder) / "corrected" / "bud_corrected_masks.tif").is_file():
        continue                                        # not corrected by hand -> nothing to learn from
    name = bt.prepare_image(folder, DATA_DIR, ASH_CHANNEL)
    if name:
        images.append(name)

test_images = [n for n in images if any(t in n for t in TEST_IMAGES)]
train_images = [n for n in images if n not in test_images]
table = bt.load_table(DATA_DIR, train_images)
rows = bt.training_rows(table, MODEL, exclude_strains=EXCLUDE_STRAINS)
print(f"{len(train_images)} training images, {len(test_images)} test images")
print(rows.kind.value_counts().to_string())

## 3. Train

In [ ]:
bt.train(rows, DATA_DIR, MODEL, OUT_FILE, epochs=EPOCHS, use_gpu=USE_GPU)

## 4. Check on the test images
Scores against your corrections: **recall** = share of budded cells paired with the right bud, **precision** = share of the predicted pairs that are right, **F1** = both combined. The shipped model is scored on the same images for comparison. Also try the weights saved every 2 epochs (`my_bud_model_<MODEL>_epochNN.pt`) – an earlier epoch is often better.


In [ ]:
def load(weights):
    return BudModel.load(MODEL, use_gpu=USE_GPU, weights_file=weights)

print("your model:   ", bt.evaluate(load(OUT_FILE), DATA_DIR, test_images))
print("shipped model:", bt.evaluate(BudModel.load(MODEL, use_gpu=USE_GPU), DATA_DIR, test_images))
# earlier epochs:
# for f in sorted(OUT_FILE.parent.glob(f"{OUT_FILE.stem}_epoch*.pt")): print(f.name, bt.evaluate(load(f), DATA_DIR, test_images))


## Use your model
Copy `models/my_bud_model_<MODEL>.pt` to a folder of your choice, rename it to `bud_model_<MODEL>.pt`
(e.g. `bud_model_with_ASH1CLB2.pt`) and point the config file to that folder:

```json
"bud_classification": {"enabled": true, "model": "with_ASH1CLB2", "weights_dir": "C:\\my_models"}
```